In [8]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import json
import time
from datetime import datetime
import re

In [9]:
def load_api_key():
    """
    Load Steam API keys from the .env file located in the .venv folder.
    
    Returns:
        tuple: A tuple containing (api_key).
    """
    # Notebook is in the 'src/' folder, so go up one level to reach '.venv/.env'
    env_path = r'..\src\config.env'
    
    load_dotenv(dotenv_path=env_path)
    api_key = os.getenv('itd_api_key')
    return api_key

api_key = load_api_key()


In [10]:
games = pd.read_json(r'..\data\games_id_all.json')
games = games.loc['apps', 'response']

def name_formatting(id):
    game_data = games[id]
    #print(games.loc['apps', 'response'][id])
    game_name = game_data['name']
    game_id = game_data['appid']

    game_name = re.sub(r'[^\x00-\x7F]+', '', game_name)
    formatted_name = game_name.lower().strip().replace('?', '').replace(',', '').replace(' - ', '-').replace('.', '').replace(': ', '-').replace(':', '-').replace("'", '').replace('!', '').replace(' (', '-').replace('(', '-').replace(')', '').replace('/', '-').replace('+', 'and').replace('&', 'and').replace('  ', ' ').replace(' ', '-')
    return formatted_name, game_id

In [11]:
def date_formatting(date, id):
    if(date == 'Coming soon' or date == 'To be announced'):
        return date
    date = date.replace(',', '').split(' ')
    #print(f'{id} {date}')
    if(len(date) < 3):
        return 'date error'
    
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    if(date[0] in months):
        for i in range(len(months)):
            if(date[0] == months[i]):
                new_date = date[1] + '-' + str(i + 1) + '-' + date[2]
    elif(date[1] in months):
        for i in range(len(months)):
            if(date[1] == months[i]):
                new_date = date[0] + '-' + str(i + 1) + '-' + date[2]
    else:
        print(date)
        return 'data error'

    
    date_format = '%d-%m-%Y'
    date_datetime = datetime.strptime(new_date, date_format)

    formatted_date = date_datetime.strftime('%Y-%m-%dT%H:%M:%S+01:00')

    return formatted_date

In [12]:
def used_ids_write(chunk, id):
    with open(rf"..\data\games_prices\used_ids_{chunk}.txt", "a", encoding="utf-8") as id_file:
        id_file.write(str(id) + "\n")

def jsonl_write(name, new_entry):
    with open(rf"..\data\games_prices\{name}", "a", encoding="utf-8") as jsonl_file:
        jsonl_file.write(json.dumps(new_entry) + "\n")

In [13]:
chunk = 7
chunk_name = f'price_data_chunk_{chunk}.jsonl'
game_chunk = f'games_chunk_{chunk}.jsonl'

game_details_cache = {}

try:
    with open(rf"..\data\games_informations\{game_chunk}", "r", encoding="utf-8") as jsonl_file:
        for line in jsonl_file:
            if line.strip():
                try:
                    line_data = json.loads(line)
                    game_details_cache.update(line_data)
                except json.JSONDecodeError:
                    pass
except FileNotFoundError:
    print(f"Warning: {game_chunk} not found")

In [14]:
batch_size = 100

last_id = -1
used_indexes = 0
try:
    with open(rf'..\data\games_prices\used_ids_{chunk}.txt', 'r', encoding='utf-8') as ids_file:
        for line in ids_file:
            line = line.strip()
            if line:
                used_indexes += 1
except FileNotFoundError:
    print("File not found, staying with default")
except ValueError:
    print("Found a non-integer in the file!")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

index = used_indexes

ids = list(game_details_cache.keys())

while index <= len(ids):
    if(index > 0 and index % 2000 == 0):
        time.sleep(60)
    if(index >= 10000):
            break

    games_names = []
    games_ids = []
    release_dates = {}

    for i in range(batch_size):
        if(index + i > 10000 or index + i >= len(ids)):
            break
        appid = ids[index + i]
        name = game_details_cache[appid]['name']
        release_date = game_details_cache[appid]['release_date']['date']

        games_ids.append(appid)
        games_names.append(name.strip())

        release_date = date_formatting(release_date, appid)
        release_dates.update({appid: release_date})
        
    index += len(games_ids)

    URL = 'https://api.isthereanydeal.com/lookup/id/title/v1'

    params = {
        'key': api_key
    }

    headers = {
        'Content-Type': 'application/json'
    }
     
    response = requests.post(URL, params=params, json=games_names, headers=headers)

    if response.status_code == 200:
        data = response.json()
    elif response.status_code == 400:
        data = {}
        counter = 0
        for game in games_names:
            if(counter > 0 and counter % 100 == 0):
                time.sleep(60)

            single_response = requests.post(URL, params=params, json=[game], headers=headers)
            if single_response.status_code == 200:
                game_json = single_response.json()
                data[game] = game_json.get(game)
            else:
                print(f"Error with game name {game}")
            counter += 1
    else:
        print(f'Error: {response.status_code}')
        index += len(games_ids)
        time.sleep(60)
        continue

    # for index, item in enumerate(games_ids):
    #     print(item)                       #game steam id
    #     print(games_names[index])         #game name
    #     print(release_dates[item])        #game release date
    #     print(data[games_names[index]])   #game itad id

    URL_prices = 'https://api.isthereanydeal.com/games/history/v2'

    for idx, id in enumerate(games_ids):

        if data.get(games_names[idx], '') != '':
            params = {
                'key': api_key,
                'shops': 61,
                'country': 'PL',
                'since': release_dates[id],
                'id': data[games_names[idx]]
            }
        else:
            new_entry = {appid: 'missing_id'}
            used_ids_write(chunk, appid)
            jsonl_write(chunk_name, new_entry)
            continue

        response = requests.get(URL_prices, params=params)

        if(response.status_code == 200):
            price_log = response.json()
            prices = []
            for i in price_log:
                timestamp = i['timestamp']
                price = i['deal']['price']['amountInt']
                prices.append({timestamp: price})
            
            new_entry = {id: prices}

            used_ids_write(chunk, id)
            jsonl_write(chunk_name, new_entry)
        else:
            new_entry = {id: 'Data_error'}
            used_ids_write(chunk, id)
            jsonl_write(chunk_name, new_entry)
            print(f'Error prices {response.status_code}')

    if index + 1 >= 10000:
        break
    time.sleep(61)

File not found, staying with default
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error prices 400
Error price

KeyboardInterrupt: 